In [ ]:
import tomllib
from pathlib import Path
import pandas as pd

RUNS = 'runs'
PREFIX = 'graphene_tilt'          # <crystal.name>_<calculation.ID> (without the _NN of a sweep)

# sweep table written by executeTB.py: one row per calculation, one column per swept key.
# A single calculation (no sweep) has no table: it is treated as a sweep of one row and no swept keys.
sweep_path = Path(RUNS) / f'{PREFIX}_sweep.txt'
if sweep_path.exists():
    sweep = pd.read_csv(sweep_path, sep='\t', comment='#')
else:
    sweep = pd.DataFrame({'index': [0], 'name': [PREFIX]})
swept_keys = [k for k in sweep.columns if k not in ('index', 'name')]

# full input of every calculation, from its runs/<name>.toml (same order as the rows of sweep)
#   e.g. configs[i]['Field']['lambda']
configs = []
for name in sweep['name']:
    with open(Path(RUNS) / f'{name}.toml', 'rb') as f:
        configs.append(tomllib.load(f))
sweep

FileNotFoundError: [Errno 2] No such file or directory: 'runs/graphene_tiltq.toml'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
#plt.style.use('/home/lplaja/.matplotlib/stylelib/obsidian.mplstyle')

def gather_data(fnamelist):
    n_columns=len(fnamelist)

    t=[]
    filename=fnamelist[0]
    with open(filename, "r") as fh:  
        for line in fh:
            if line.startswith("#"):
                continue
            it, ix, iy = map(complex, line.split())
            t.append(np.real(it))

    t=np.array(t)

    n_rows=len(t)

    dipole_vel_x=np.zeros((n_rows, n_columns))
    dipole_vel_y=np.zeros((n_rows, n_columns))

    for i_file, filename in enumerate(fnamelist):

        ivel_x=[]
        ivel_y=[]
        with open(filename, "r") as fh:
            for line in fh:
                if line.startswith("#"):
                    continue
                it, ix, iy = map(complex, line.split())
                ivel_x.append(np.real(ix))
                ivel_y.append(np.real(iy))
# Caso de cambio de nt
#        if i_file==2:
#            dipole_vel_x[:,i_file]=ivel_x[::2]
#            dipole_vel_y[:,i_file]=ivel_y[::2]
#        else:
#            dipole_vel_x[:,i_file]=ivel_x
#            dipole_vel_y[:,i_file]=ivel_y
# caso dsin cambio de nt  
        dipole_vel_x[:,i_file]=ivel_x
        dipole_vel_y[:,i_file]=ivel_y       

    # dipole acceleration per unit cell, d(vd)/dt with t in s: C m/s^2
    accel_x=np.gradient(dipole_vel_x, t, axis=0)
    accel_y=np.gradient(dipole_vel_y, t, axis=0)
    accel_x=accel_x[2:-3]
    accel_y=accel_y[2:-3]

    return t, dipole_vel_x, dipole_vel_y, accel_x, accel_y

#fnamelist=['graphene_prueba_dipole_velocity_nk300.txt','graphene_prueba_dipole_velocity_nk500.txt',
#           'graphene_prueba_dipole_velocity_nk900.txt']
#labels=['nk=300','nk=500','nk=900']

#fnamelist=['graphene_tilt_oscar_0_dipole_velocity.txt','graphene_tilt_oscar_15_dipole_velocity.txt', 
#           'graphene_tilt_oscar_30_dipole_velocity.txt']
#labels=['theta=0','theta=15','theta=30']

# files and labels from the sweep table (cell above)
fnamelist=[f"{RUNS}/{name}_dipole_velocity.txt" for name in sweep['name']]
labels=[", ".join(f"{k}={r[k]:.4g}" if isinstance(r[k], float) else f"{k}={r[k]}" for k in swept_keys)
           or r['name']                    # no sweep: label = name of the calculation
        for _, r in sweep.iterrows()]
# Oscar's tilt instead of chi (tilt = 90 deg - chi):
# labels=[f"tilt={90-np.rad2deg(c):.0f}" for c in sweep['Field.chi']]


t, dipole_vel_x, dipole_vel_y, accel_x, accel_y=gather_data(fnamelist)

nt=t.size-5
imask=int(nt/10)
mask=np.ones(nt)
igauss=np.arange(0,imask)
print(f'{igauss.shape=}   {mask.shape=}')
print(f'{nt=}   {imask=}')
print(f'{accel_x.shape=}   {accel_y.shape=}')
mask[nt-imask:]*=np.exp(-(igauss/10)**2)
for nn in range(len(fnamelist)):
    accel_x[:,nn]*=mask
    accel_y[:,nn]*=mask


# Fourier transform as an integral, a(w) = int a(t) exp(-i w t) dt ~ dt * FFT:  C m/s
dt=t[1]-t[0]
FT_npts=2*accel_x.shape[0]//8
FT_accel_x=np.fft.fft(accel_x, axis=0)[:FT_npts]*dt
FT_accel_y=np.fft.fft(accel_y, axis=0)[:FT_npts]*dt


plt.rcParams['legend.fontsize'] = 8

fig, ax=plt.subplots(1,1, figsize=(6,4))

#ax[0,0].plot(t, np.real(dipole_vel_y), label=labels)
#ax[0,0].set_xlabel("t")
#ax[0,0].set_ylabel("dipole velocity")
#ax[0,0].set_title("Dipole velocity y")
#ax[0,0].legend()
#ax[0,0].grid(False)

#ax[0,1].plot(t[2:-3], np.real(accel_y), label=labels)
#ax[0,1].set_xlabel("t")
#ax[0,1].set_ylabel("acceleration")
#ax[0,1].set_title("Acceleration y")
#ax[0,1].legend()
#ax[0,1].grid(False)

# ax[0].semilogy(np.abs(FT_accel_x)**2+np.abs(FT_accel_y)**2, label=labels)
# ax[0].set_xlabel("")
# ax[0].set_ylabel("acceleration")
# ax[0].set_title("FT Acceleration")
# ax[0].legend()
# ax[0].grid(False)

# harmonic order: FFT index / duration of the record in optical periods (was a hardcoded 8)
periods = {c['time']['tfin'] - c['time']['tini'] for c in configs}
assert len(periods) == 1, "all calculations must have the same time window"
w = np.arange(len(FT_accel_y))/periods.pop()
spectrum=np.abs(FT_accel_x)**2+np.abs(FT_accel_y)**2          # C^2 m^2 / s^2
ax.semilogy(w, spectrum, label=labels)
ax.set_xlabel("harmonic order")
ax.set_ylabel(r"$|\ddot d_x(\omega)|^2+|\ddot d_y(\omega)|^2$  (C$^2$ m$^2$ s$^{-2}$)")
ax.set_title("HHG spectrum (dipole acceleration per unit cell)")
ax.legend()
ax.set_xlim(0,20)
ax.set_ylim(spectrum.max()*1e-10, spectrum.max()*10)   
ax.set_xticks(np.arange(1, 20, 2))        # 10 decades below the maximum
ax.grid(True, linestyle='dotted')

plt.tight_layout()

In [ ]:
# ---- harmonic analysis: Stokes parameters and polarization ellipse of every harmonic order
import sys
repo_src = next(p/'src' for p in Path.cwd().resolve().parents if (p/'src'/'Field.py').exists())
sys.path.insert(0, str(repo_src))
import Field                                    # ellipse_to_jones: same (phi, delta_varphi) convention as PulsedField

HARM_WIDTH = 1.0                                # window [q - width/2, q + width/2] in harmonic orders (1 or 2)
Q_VALUES = np.arange(1, 20, 2)                  # harmonic orders analysed (odd)

def harmonic_table(FTx, FTy, w, q_values, width):
    """One row per (calculation, harmonic q) with the spectral integral and the averaged polarization.

    FTx, FTy: numpy FFT (x dt) of the acceleration, shape (n_w, n_calc); w: harmonic order of each row.
    numpy's FFT is int a(t) exp(-i w t) dt: for a(t) = Re[A exp(-i w0 t)] (convention of Field.py and of
    the slides) it gives conj(A)/2 at +w0, so a_par = conj(FTx), a_perp = conj(FTy)  (par = x, perp = y).
    Stokes parameters (slide 48), per frequency:
        S0 = |a_par|^2 + |a_perp|^2      S1 = |a_par|^2 - |a_perp|^2
        S2 = 2 Re(a_par a_perp^*)        S3 = 2 Im(a_par a_perp^*)  = i (a_perp a_par^* - a_perp^* a_par)
    In the window: I = sum S0 dq (integral in harmonic orders), s_i = sum S_i / sum S0 (average weighted with
    |FT|^2), P = |(s1, s2, s3)| (degree of polarization), and the ellipse of the polarized part
        chi = atan2(s2, s1)/2,   eps = tan(arcsin(s3/P)/2),   (phi, delta_varphi) = Field.ellipse_to_jones(chi, eps)
    """
    a_par, a_perp = np.conj(FTx), np.conj(FTy)
    S0 = np.abs(a_par)**2 + np.abs(a_perp)**2
    S1 = np.abs(a_par)**2 - np.abs(a_perp)**2
    S2 = 2*np.real(a_par*np.conj(a_perp))
    S3 = 2*np.imag(a_par*np.conj(a_perp))
    dq = w[1] - w[0]

    rows = []
    for i, r in sweep.iterrows():
        for q in q_values:
            win = (w >= q - width/2) & (w < q + width/2)
            I = S0[win, i].sum()*dq
            s1, s2, s3 = (S[win, i].sum()*dq/I for S in (S1, S2, S3))
            P = np.sqrt(s1**2 + s2**2 + s3**2)
            chi = 0.5*np.arctan2(s2, s1)
            eps = np.tan(0.5*np.arcsin(np.clip(s3/P, -1, 1)))
            phi, delta_varphi = Field.ellipse_to_jones(chi, eps)
            rows.append({'name': r['name'], **{k: r[k] for k in swept_keys}, 'q': q, 'I': I,
                         'S1': s1, 'S2': s2, 'S3': s3, 'P': P, 'chi': chi, 'eps': eps,
                         'phi': phi, 'delta_varphi': delta_varphi})
    return pd.DataFrame(rows)

harm = harmonic_table(FT_accel_x, FT_accel_y, w, Q_VALUES, HARM_WIDTH)
harm


In [ ]:
# ---- per-harmonic plots: one line per calculation of the sweep
I_REL_MIN = 1e-8        # polarization not plotted where I_q < I_REL_MIN * max_q I_q (no signal, only numerical noise)
fig, axs = plt.subplots(2, 2, figsize=(9, 6), sharex=True)
for (name, g), lab in zip(harm.groupby('name', sort=False), labels):
    axs[0, 0].semilogy(g['q'], g['I'], 'o-', label=lab)
    g = g[g['I'] > I_REL_MIN*g['I'].max()]
    axs[0, 1].plot(g['q'], np.rad2deg(g['chi']), 'o-')
    axs[1, 0].plot(g['q'], g['eps'], 'o-')
    axs[1, 1].plot(g['q'], g['P'], 'o-')
axs[0, 0].set_ylabel(r"$I_q=\int_q (|\ddot d_x|^2+|\ddot d_y|^2)\,dq$  (C$^2$ m$^2$ s$^{-2}$)")
axs[0, 1].set_ylabel(r"$\chi$ (deg)");  axs[0, 1].set_ylim(-90, 90);  axs[0, 1].set_yticks(np.arange(-90, 91, 30))
axs[1, 0].set_ylabel(r"$\varepsilon$");  axs[1, 0].set_ylim(-1.05, 1.05)
axs[1, 1].set_ylabel("P (degree of polarization)");  axs[1, 1].set_ylim(0, 1.05)
for ax in axs.flat:
    ax.set_xticks(Q_VALUES);  ax.grid(True, linestyle='dotted')
for ax in axs[1]:
    ax.set_xlabel("harmonic order q")
axs[0, 0].legend(fontsize=7)
fig.suptitle(f"window width {HARM_WIDTH} harmonic orders")
plt.tight_layout()


In [ ]:
# ---- ellipse of every harmonic vs the tilt of the driver (one line per harmonic q)
DRIVER_KEY = 'Field.chi'            # swept key used as horizontal axis (driver tilt, rad)
assert DRIVER_KEY in swept_keys, f"{DRIVER_KEY} is not swept: nothing to plot against"

h = harm.copy()
h = h[h['I'] > I_REL_MIN*h.groupby('name')['I'].transform('max')]   # only harmonics with signal
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for q, g in h.groupby('q'):
    g = g.sort_values(DRIVER_KEY)
    chi_q = 0.5*np.unwrap(2*g['chi'])            # continuous along the sweep (chi is defined mod pi)
    x = np.rad2deg(g[DRIVER_KEY])
    axs[0].plot(x, np.rad2deg(chi_q), 'o-', label=f'q={q}')
    axs[1].plot(x, g['eps'], 'o-', label=f'q={q}')

x_drv = np.rad2deg(np.sort(sweep[DRIVER_KEY]))    # driver polarization, for reference
eps_drv = [c['Field']['ellip'] for c in configs]
axs[0].plot(x_drv, x_drv, 'k--', lw=1, label='driver')
axs[1].plot(np.rad2deg(sweep[DRIVER_KEY]), eps_drv, 'k--', lw=1, label='driver')

axs[0].set_ylabel(r"$\chi_q$ (deg)")
axs[1].set_ylabel(r"$\varepsilon_q$");  axs[1].set_ylim(-1.05, 1.05)
for ax in axs:
    ax.set_xlabel(r"driver $\chi$ (deg)");  ax.grid(True, linestyle='dotted')
axs[1].legend(fontsize=7)
fig.suptitle(f"harmonic polarization vs driver tilt (window width {HARM_WIDTH})")
plt.tight_layout()


![image.png](attachment:image.png)